# Outlook PO Email Reader

This notebook reads Outlook emails from `.msg` files, classifies them to identify Purchase Order (PO) emails, and extracts key details from the email body.

## 1. Install and Import Required Libraries

In [ ]:
# Install required packages (uncomment if needed)
# !pip install extract-msg pandas

In [ ]:
import extract_msg
import pandas as pd
import os
import re
from pathlib import Path
from datetime import datetime
import glob

print("Libraries imported successfully!")

## 2. Email Reading Functions

In [ ]:
def read_msg_file(msg_path):
    """
    Read an Outlook .msg file and extract email details.
    
    Args:
        msg_path: Path to the .msg file
    
    Returns:
        Dictionary containing email details
    """
    try:
        # Skip empty files
        if os.path.getsize(msg_path) == 0:
            print(f"  ⚠ Skipping empty file: {os.path.basename(msg_path)}")
            return None
        
        msg = extract_msg.Message(msg_path)
        
        # Get attachment names safely
        attachment_names = []
        for att in msg.attachments:
            if hasattr(att, 'longFilename') and att.longFilename:
                attachment_names.append(att.longFilename)
            elif hasattr(att, 'name') and att.name:
                attachment_names.append(att.name)
        
        email_data = {
            'file_name': os.path.basename(msg_path),
            'subject': msg.subject or '',
            'sender': msg.sender or '',
            'sender_email': '',  # Will be parsed from sender string
            'to': str(msg.to) if msg.to else '',
            'cc': str(msg.cc) if msg.cc else '',
            'bcc': str(msg.bcc) if msg.bcc else '',
            'date': str(msg.date) if msg.date else '',
            'body': msg.body or '',
            'html_body': '',
            'attachments': attachment_names,
            'attachment_count': len(msg.attachments)
        }
        
        # Parse sender email from sender string (format: "Name <email@example.com>")
        sender_str = msg.sender or ''
        email_match = re.search(r'<([^>]+)>', sender_str)
        if email_match:
            email_data['sender_email'] = email_match.group(1)
        
        msg.close()
        return email_data
    
    except Exception as e:
        print(f"Error reading {msg_path}: {str(e)}")
        return None


def read_all_msg_files(folder_path):
    """
    Read all .msg files from a folder.
    
    Args:
        folder_path: Path to folder containing .msg files
    
    Returns:
        List of email data dictionaries
    """
    emails = []
    msg_files = glob.glob(os.path.join(folder_path, '*.msg'))
    
    print(f"Found {len(msg_files)} .msg files in {folder_path}")
    
    for msg_file in msg_files:
        email_data = read_msg_file(msg_file)
        if email_data:
            emails.append(email_data)
            print(f"  ✓ Read: {email_data['file_name']}")
    
    return emails


print("Email reading functions defined!")

## 3. PO Email Classification

These functions classify emails to identify Purchase Order related emails.

In [ ]:
# Keywords and patterns for PO email classification
PO_KEYWORDS = [
    # PO identifiers
    'purchase order', 'po#', 'po number', 'p.o.', 'p.o', 'po:', 'po ',
    # Order-related terms
    'order confirmation', 'order acknowledgment', 'order acknowledgement',
    'order placed', 'new order', 'order details', 'order number',
    # Procurement terms
    'procurement', 'requisition', 'indent', 'supply order',
    # Commercial terms
    'quotation', 'quote', 'invoice', 'proforma', 'pro-forma',
    # Processing terms
    'packing', 'shipment', 'delivery', 'dispatch', 'trims',
    # Action terms related to PO
    'please confirm', 'kindly confirm', 'attached po', 'attached purchase order',
    # Specific PO formats
    'mel2025po', 'mel2024po', 'mel2026po'  # MEL PO format from your files
]

# Regex patterns for PO numbers
PO_NUMBER_PATTERNS = [
    r'P[O0][#:\s-]*([A-Z0-9-]+)',           # PO#12345, PO: 12345, PO-12345
    r'MEL\d{4}PO\d+',                       # MEL2025PO12232 format
    r'Purchase\s*Order[#:\s-]*([A-Z0-9-]+)', # Purchase Order #12345
    r'Order\s*(?:No|Number|#)[.:\s-]*([A-Z0-9-]+)',  # Order No. 12345
    r'[A-Z]{2,4}\d{4}PO\d+',                # Generic XXX2024PO12345 format
]


def calculate_po_score(email_data):
    """
    Calculate a confidence score for whether an email is PO-related.
    
    Args:
        email_data: Dictionary containing email details
    
    Returns:
        Tuple of (score, matched_keywords, matched_patterns)
    """
    score = 0
    matched_keywords = []
    matched_patterns = []
    
    # Combine subject and body for searching
    text = f"{email_data['subject']} {email_data['body']}".lower()
    subject = email_data['subject'].lower()
    
    # Check for keywords
    for keyword in PO_KEYWORDS:
        if keyword.lower() in text:
            # Higher weight for keywords in subject
            if keyword.lower() in subject:
                score += 3
            else:
                score += 1
            matched_keywords.append(keyword)
    
    # Check for PO number patterns (case insensitive)
    for pattern in PO_NUMBER_PATTERNS:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            score += 5  # Strong indicator
            matched_patterns.extend(matches if isinstance(matches[0], str) else [m[0] for m in matches])
    
    # Check filename for PO indicators
    filename = email_data['file_name'].lower()
    if 'po' in filename or 'purchase' in filename or 'order' in filename:
        score += 2
    
    # Check attachments for PO documents
    for attachment in email_data['attachments']:
        att_lower = attachment.lower()
        if 'po' in att_lower or 'purchase' in att_lower or 'order' in att_lower:
            score += 2
    
    return score, list(set(matched_keywords)), list(set(matched_patterns))


def classify_email(email_data, threshold=5):
    """
    Classify an email as PO-related or not.
    
    Args:
        email_data: Dictionary containing email details
        threshold: Minimum score to classify as PO email (default: 5)
    
    Returns:
        Dictionary with classification results
    """
    score, keywords, patterns = calculate_po_score(email_data)
    
    is_po_email = score >= threshold
    
    # Determine confidence level
    if score >= 15:
        confidence = 'HIGH'
    elif score >= 10:
        confidence = 'MEDIUM'
    elif score >= threshold:
        confidence = 'LOW'
    else:
        confidence = 'NOT_PO'
    
    return {
        'is_po_email': is_po_email,
        'confidence': confidence,
        'score': score,
        'matched_keywords': keywords,
        'matched_patterns': patterns
    }


def classify_all_emails(emails, threshold=5):
    """
    Classify all emails and separate PO emails from non-PO emails.
    
    Args:
        emails: List of email data dictionaries
        threshold: Minimum score to classify as PO email
    
    Returns:
        Tuple of (po_emails, non_po_emails)
    """
    po_emails = []
    non_po_emails = []
    
    for email in emails:
        classification = classify_email(email, threshold)
        email['classification'] = classification
        
        if classification['is_po_email']:
            po_emails.append(email)
            print(f"  ✓ PO Email ({classification['confidence']}): {email['subject'][:50]}...")
        else:
            non_po_emails.append(email)
            print(f"  ✗ Not PO: {email['subject'][:50]}...")
    
    print(f"\nClassification complete: {len(po_emails)} PO emails, {len(non_po_emails)} non-PO emails")
    
    return po_emails, non_po_emails


print("Email classification functions defined!")

## 4. PO Detail Extraction

These functions extract key details from PO emails.

In [ ]:
def extract_po_from_label(text):
    """
    Extract PO number specifically from "Purchase Order Number" label.
    This function prioritizes the structured PO number label.
    
    Args:
        text: Email body or text to search
    
    Returns:
        PO number string or None if not found
    """
    # Pattern to match "Purchase Order Number" followed by the PO number
    # Handles various formats: "Purchase Order Number: MEL2026PO14536"
    pattern = r'Purchase\s*Order\s*Number[:\s-]*([A-Z0-9-]+)'
    match = re.search(pattern, text, re.IGNORECASE)
    
    if match:
        po = match.group(1).strip()
        # Clean up - remove any trailing non-alphanumeric characters
        po = re.sub(r'[^A-Z0-9-].*$', '', po, flags=re.IGNORECASE)
        return po if po else None
    return None


def get_primary_po_number(text):
    """
    Extract the single, most reliable PO number from text.
    Priority: Label > MEL format > Other formats
    
    Args:
        text: Email body or text to search
    
    Returns:
        Primary PO number string or None if not found
    """
    # Priority 1: Extract from "Purchase Order Number" label
    po_from_label = extract_po_from_label(text)
    if po_from_label:
        return po_from_label
    
    # Priority 2: MEL format (most reliable structured format)
    mel_pattern = r'MEL\d{4}PO\d+'
    mel_matches = re.findall(mel_pattern, text, re.IGNORECASE)
    if mel_matches:
        # Return the first valid MEL format (clean match)
        for match in mel_matches:
            # Ensure it's not malformed (no prefix like EMEL, RMEL)
            if match.startswith('MEL'):
                return match
    
    # Priority 3: Generic PO# format
    po_hash = r'P[O0]#\s*([A-Z0-9-]+)'
    po_hash_matches = re.findall(po_hash, text, re.IGNORECASE)
    if po_hash_matches:
        return po_hash_matches[0].strip()
    
    # Priority 4: PO: or PO format with numbers (less reliable, filter short matches)
    po_colon = r'P[O0][:\s]+([A-Z0-9-]+)'
    matches = re.findall(po_colon, text, re.IGNORECASE)
    for match in matches:
        if len(match) > 3:
            return match.strip()
    
    return None


def extract_po_numbers(text):
    """
    Extract primary PO number from text (returns single best match, not all matches).
    Use get_primary_po_number() for reliable single PO extraction.
    
    Args:
        text: Email body or text to search
    
    Returns:
        List containing the primary PO number if found, empty list otherwise
    """
    primary_po = get_primary_po_number(text)
    return [primary_po] if primary_po else []


def extract_po_details(email_data):
    """
    Extract all key PO details from an email.
    
    Args:
        email_data: Dictionary containing email details
    
    Returns:
        Dictionary with extracted PO details
    """
    # Combine subject and body
    full_text = f"{email_data['subject']} {email_data['body']}"
    
    # Also check filename for codes
    full_text_with_filename = f"{full_text} {email_data['file_name']}"
    
    # Extract PO number from the label first (if it exists)
    po_from_label = extract_po_from_label(email_data['body'])
    
    po_details = {
        'file_name': email_data['file_name'],
        'subject': email_data['subject'],
        'sender': email_data['sender'],
        'sender_email': email_data['sender_email'],
        'email_date': email_data['date'],
        'po_number_from_label': po_from_label,  # NEW: PO from "Purchase Order Number" label
        'po_numbers': extract_po_numbers(full_text_with_filename),
        'item_codes': extract_item_codes(full_text_with_filename),
        'dates_mentioned': extract_dates(full_text),
        'amounts': extract_amounts(full_text),
        'quantities': extract_quantities(full_text),
        'attachments': email_data['attachments'],
        'attachment_count': email_data['attachment_count'],
        'confidence': email_data.get('classification', {}).get('confidence', 'N/A'),
        'classification_score': email_data.get('classification', {}).get('score', 0),
        'body_preview': email_data['body'][:500] if email_data['body'] else ''
    }
    
    return po_details


## 5. Export Functions

In [ ]:
def export_to_csv(po_details, output_path):
    """
    Export PO details to a CSV file.
    
    Args:
        po_details: List of PO detail dictionaries
        output_path: Path to save the CSV file
    """
    # Flatten the data for CSV
    flattened_data = []
    
    for detail in po_details:
        # Get the primary PO number (prefer label version if available, otherwise use extracted)
        primary_po = detail['po_number_from_label'] or (detail['po_numbers'][0] if detail['po_numbers'] else None)
        
        flat_record = {
            'File Name': detail['file_name'],
            'Subject': detail['subject'],
            'Sender': detail['sender'],
            'Sender Email': detail['sender_email'],
            'Email Date': detail['email_date'],
            'Primary PO Number': primary_po,  # Single best PO number
            'Item Codes': '; '.join(detail['item_codes']),
            'Dates Mentioned': '; '.join(detail['dates_mentioned']),
            'Amounts': '; '.join(detail['amounts']),
            'Quantities': '; '.join(detail['quantities']),
            'Attachments': '; '.join(detail['attachments']),
            'Attachment Count': detail['attachment_count'],
            'Confidence': detail['confidence'],
            'Classification Score': detail['classification_score'],
            'Body Preview': detail['body_preview'][:200]
        }
        flattened_data.append(flat_record)
    
    df = pd.DataFrame(flattened_data)
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    print(f"\nExported {len(po_details)} PO emails to: {output_path}")
    return df


## 6. Main Processing Pipeline

In [ ]:
# Configuration
OUTLOOK_FOLDER = r'D:\Year 4 Sem 1\PO\PO\Outlook'
OUTPUT_FOLDER = r'D:\Year 4 Sem 1\PO\PO\output'
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, 'outlook_po_emails.csv')

# Classification threshold (lower = more emails classified as PO)
CLASSIFICATION_THRESHOLD = 5

print(f"Outlook folder: {OUTLOOK_FOLDER}")
print(f"Output file: {OUTPUT_FILE}")

In [ ]:
# Step 1: Read all emails from the Outlook folder
print("Step 1: Reading emails...")
print("-" * 40)
emails = read_all_msg_files(OUTLOOK_FOLDER)

In [ ]:
# Step 2: Classify emails
print("\nStep 2: Classifying emails...")
print("-" * 40)
po_emails, non_po_emails = classify_all_emails(emails, threshold=CLASSIFICATION_THRESHOLD)

In [ ]:
# Step 3: Extract PO details from classified PO emails
print("\nStep 3: Extracting PO details...")
print("-" * 40)
po_details = extract_all_po_details(po_emails)

In [ ]:
# Step 4: Export results
print("\nStep 4: Exporting results...")
print("-" * 40)
df = export_to_csv(po_details, OUTPUT_FILE)

# Generate summary report
generate_summary_report(po_emails, non_po_emails, po_details)

In [ ]:
# Display the results DataFrame
if len(po_details) > 0:
    display(df)
else:
    print("No PO emails found.")

## 7. Inspect Individual Email Details

In [ ]:
# Inspect a specific email in detail
def inspect_email(email_index):
    """
    Display detailed information about a specific PO email.
    """
    if email_index >= len(po_details):
        print(f"Invalid index. Max index is {len(po_details) - 1}")
        return
    
    detail = po_details[email_index]
    email = po_emails[email_index]
    
    print(f"{'='*60}")
    print(f"EMAIL #{email_index}")
    print(f"{'='*60}")
    print(f"\nFile: {detail['file_name']}")
    print(f"Subject: {detail['subject']}")
    print(f"From: {detail['sender']} <{detail['sender_email']}>")
    print(f"Date: {detail['email_date']}")
    print(f"\nClassification: {detail['confidence']} (score: {detail['classification_score']})")
    print(f"\nPO Numbers: {detail['po_numbers']}")
    print(f"Item Codes: {detail['item_codes']}")
    print(f"Amounts: {detail['amounts']}")
    print(f"Quantities: {detail['quantities']}")
    print(f"Dates: {detail['dates_mentioned']}")
    print(f"\nAttachments ({detail['attachment_count']}):")
    for att in detail['attachments']:
        print(f"  - {att}")
    print(f"\nEmail Body Preview:")
    print("-" * 40)
    print(email['body'][:1000] if email['body'] else '(No body)')
    print("-" * 40)

# Example: Inspect first PO email
if len(po_details) > 0:
    inspect_email(0)